In [ ]:
#| default_exp data

# Data

> Google Cloud Storage, Firestore, Cloud SQL PostgreSQL, and Memorystore Redis.

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
try:
    from google.cloud import storage
    from google.cloud import firestore as fs
    from google.cloud.sql.connector import Connector as SQLConnector
    from google.cloud import redis_v1 as redis_client
    from google.cloud.redis_v1 import CloudRedisClient, Instance
    import googleapiclient.discovery
except ImportError:
    pass

## Google Cloud Storage

In [ ]:
#| export
def _gcs(auth):
    return storage.Client(project=auth.project, credentials=auth.credentials)


def create_bucket(
    auth,
    name: str,
    location: str = None,
    versioning: bool = True,
    labels: dict = None,
    **compliance_opts,
) -> dict:
    """Create or update a GCS bucket with uniform access, versioning, and encryption."""
    client = _gcs(auth)
    location = location or auth.region
    try:
        bucket = client.get_bucket(name)
    except Exception:
        bucket = client.bucket(name)
        bucket.location = location
        bucket = client.create_bucket(bucket)

    bucket.iam_configuration.uniform_bucket_level_access_enabled = True
    if versioning:
        bucket.versioning_enabled = True
    if labels:
        bucket.labels = labels
    bucket.patch()
    return {'name': bucket.name, 'location': bucket.location, 'url': f'gs://{bucket.name}'}


def bucket_url(name: str, key: str = '') -> str:
    "Return the gs:// URL for a bucket or object."
    return f'gs://{name}/{key}'.rstrip('/')


def signed_url(auth, name: str, key: str, hours: int = 1) -> str:
    """Generate a signed URL for temporary object access."""
    import datetime
    client = _gcs(auth)
    blob = client.bucket(name).blob(key)
    return blob.generate_signed_url(
        expiration=datetime.timedelta(hours=hours),
        method='GET',
        credentials=auth.credentials,
    )


def bucket_conn(name: str) -> str:
    "Return a gs:// connection URI for the bucket."
    return f'gs://{name}'

## Firestore

In [ ]:
#| export
def _firestore(auth):
    return fs.Client(project=auth.project, credentials=auth.credentials)


def create_collection(auth, name: str) -> str:
    """Return a Firestore collection reference (creates on first write; idempotent)."""
    # Firestore collections are schema-free and created on first document write.
    # We write a placeholder doc to materialise the collection.
    client = _firestore(auth)
    doc_ref = client.collection(name).document('__init__')
    doc_ref.set({'gcpeasy': 'collection initialised'}, merge=True)
    return name


def firestore_conn(auth) -> str:
    "Return a firestore:// URI for the project database."
    return f'firestore://{auth.project}/(default)'

## Cloud SQL PostgreSQL

In [ ]:
#| export
def create_postgres(
    auth,
    name: str,
    tier: str = 'db-g1-small',
    engine_version: str = 'POSTGRES_16',
    master_username: str = 'pgadmin',
    master_password: str = None,
    deletion_protection: bool = False,
    backup_retention: int = 7,
    labels: dict = None,
    **compliance_opts,
) -> dict:
    """Create or update a Cloud SQL PostgreSQL instance.

    SSL is always required; at-rest encryption is GCP-default.
    Pass `deletion_protection=True` with HIPAA/compliance profiles.
    """
    import secrets as _sec
    sqladmin = googleapiclient.discovery.build(
        'sqladmin', 'v1beta4', credentials=auth.credentials
    )
    password = master_password or _sec.token_urlsafe(24)
    body = {
        'name': name,
        'region': auth.region,
        'databaseVersion': engine_version,
        'settings': {
            'tier': tier,
            'ipConfiguration': {'requireSsl': True},
            'backupConfiguration': {
                'enabled': True,
                'transactionLogRetentionDays': backup_retention,
            },
            'deletionProtectionEnabled': deletion_protection,
            'userLabels': labels or {},
        },
    }
    try:
        sqladmin.instances().get(
            project=auth.project, instance=name
        ).execute()
    except Exception:
        op = sqladmin.instances().insert(
            project=auth.project, body=body
        ).execute()
        return {'name': name, 'operation': op.get('name'), 'password': password}
    return {'name': name}


def postgres_conn(auth, name: str, db: str = 'postgres') -> str:
    "Return a Cloud SQL connection string for use with cloud-sql-python-connector."
    return f'{auth.project}:{auth.region}:{name}'

## Memorystore Redis

In [ ]:
#| export
def create_redis(
    auth,
    name: str,
    tier: str = 'BASIC',
    memory_size_gb: int = 1,
    redis_version: str = 'REDIS_7_0',
    transit_encryption: bool = True,
    labels: dict = None,
    **compliance_opts,
) -> dict:
    """Create or update a Memorystore Redis instance.

    In-transit encryption is enabled by default (`transit_encryption=True`).
    At-rest encryption uses Google-managed keys by default.
    """
    client = CloudRedisClient(credentials=auth.credentials)
    parent = f'projects/{auth.project}/locations/{auth.region}'
    instance_name = f'{parent}/instances/{name}'

    try:
        existing = client.get_instance(name=instance_name)
        return {'name': existing.name, 'host': existing.host, 'port': existing.port}
    except Exception:
        pass

    instance = Instance(
        display_name=name,
        tier=Instance.Tier[tier],
        memory_size_gb=memory_size_gb,
        redis_version=redis_version,
        transit_encryption_mode=(
            Instance.TransitEncryptionMode.SERVER_AUTHENTICATION
            if transit_encryption
            else Instance.TransitEncryptionMode.DISABLED
        ),
        labels=labels or {},
    )
    op = client.create_instance(parent=parent, instance_id=name, instance=instance)
    result = op.result(timeout=600)
    return {'name': result.name, 'host': result.host, 'port': result.port}


def redis_conn(auth, name: str) -> str:
    "Return a redis:// URI for the Memorystore instance."
    client = CloudRedisClient(credentials=auth.credentials)
    instance = client.get_instance(
        name=f'projects/{auth.project}/locations/{auth.region}/instances/{name}'
    )
    return f'redis://{instance.host}:{instance.port}'